# EXP-004: 멀티 모델 앙상블 (XGBoost + LightGBM + CatBoost)

**현재 최고:** XGBoost OOF 0.96812 / LB 0.96597  
**목표:** 3모델 앙상블로 성능 향상

| 단계 | 내용 |
|------|------|
| A | LightGBM 단일 |
| B | CatBoost 단일 |
| C | 3모델 가중 블렌딩 |

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
import warnings
warnings.filterwarnings('ignore')

mpl.rcParams['font.family'] = 'Malgun Gothic'
mpl.rcParams['axes.unicode_minus'] = False

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

TARGET = 'Irrigation_Need'
CAT_COLS = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
            'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']

target_le = LabelEncoder()
y = target_le.fit_transform(train[TARGET])

for col in CAT_COLS:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]])
    le.fit(combined)
    train[col] = le.transform(train[col])
    test[col] = le.transform(test[col])

drop_cols = ['id', TARGET]
feature_cols = [c for c in train.columns if c not in drop_cols]
X = train[feature_cols]
X_test = test[feature_cols]
sample_weights = compute_sample_weight('balanced', y)

N_FOLDS = 10
SEED = 42

print(f"피처: {len(feature_cols)}개, Train: {X.shape}, Test: {X_test.shape}")

피처: 19개, Train: (630000, 19), Test: (270000, 19)


## XGBoost (EXP-003C 최적 설정 재현)

In [2]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# --- XGBoost ---
xgb_params = {
    'n_estimators': 5000,
    'max_depth': 6,
    'learning_rate': 0.02,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 5,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'objective': 'multi:softprob',
    'num_class': 3,
    'eval_metric': 'mlogloss',
    'tree_method': 'hist',
    'device': 'cuda',
    'random_state': SEED,
    'n_jobs': -1,
    'verbosity': 0,
    'early_stopping_rounds': 200,
}

oof_xgb = np.zeros((len(X), 3))
test_xgb = np.zeros((len(X_test), 3))

print("=== XGBoost ===")
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    model = XGBClassifier(**xgb_params)
    model.fit(X.iloc[tr_idx], y[tr_idx], sample_weight=sample_weights[tr_idx],
              eval_set=[(X.iloc[va_idx], y[va_idx])], verbose=0)
    oof_xgb[va_idx] = model.predict_proba(X.iloc[va_idx])
    test_xgb += model.predict_proba(X_test) / N_FOLDS
    score = balanced_accuracy_score(y[va_idx], oof_xgb[va_idx].argmax(axis=1))
    print(f"  Fold {fold}: {score:.5f}")

xgb_oof_score = balanced_accuracy_score(y, oof_xgb.argmax(axis=1))
print(f"  XGBoost OOF: {xgb_oof_score:.5f}")

=== XGBoost ===
  Fold 0: 0.96696
  Fold 1: 0.96795
  Fold 2: 0.96728
  Fold 3: 0.96998
  Fold 4: 0.96866
  Fold 5: 0.96912
  Fold 6: 0.96920
  Fold 7: 0.96617
  Fold 8: 0.96864
  Fold 9: 0.96698
  XGBoost OOF: 0.96809


## LightGBM

In [3]:
import lightgbm as lgb

lgb_params = {
    'n_estimators': 3000,
    'max_depth': -1,
    'num_leaves': 127,
    'learning_rate': 0.04,
    'subsample': 0.8,
    'colsample_bytree': 0.6,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'min_child_samples': 20,
    'objective': 'multiclass',
    'num_class': 3,
    'metric': 'multi_logloss',
    'device': 'gpu',
    'random_state': SEED,
    'n_jobs': -1,
    'verbosity': -1,
}

oof_lgb = np.zeros((len(X), 3))
test_lgb = np.zeros((len(X_test), 3))

print("=== LightGBM ===")
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    model = LGBMClassifier(**lgb_params)
    model.fit(X.iloc[tr_idx], y[tr_idx], sample_weight=sample_weights[tr_idx],
              eval_set=[(X.iloc[va_idx], y[va_idx])],
              callbacks=[lgb.early_stopping(150, verbose=False)])
    oof_lgb[va_idx] = model.predict_proba(X.iloc[va_idx])
    test_lgb += model.predict_proba(X_test) / N_FOLDS
    score = balanced_accuracy_score(y[va_idx], oof_lgb[va_idx].argmax(axis=1))
    print(f"  Fold {fold}: {score:.5f} (best_iter: {model.best_iteration_})")

lgb_oof_score = balanced_accuracy_score(y, oof_lgb.argmax(axis=1))
print(f"  LightGBM OOF: {lgb_oof_score:.5f}")

=== LightGBM ===
  Fold 0: 0.96459 (best_iter: 853)
  Fold 1: 0.96515 (best_iter: 878)
  Fold 2: 0.96596 (best_iter: 853)
  Fold 3: 0.96817 (best_iter: 877)
  Fold 4: 0.96621 (best_iter: 878)
  Fold 5: 0.96723 (best_iter: 922)
  Fold 6: 0.96559 (best_iter: 927)
  Fold 7: 0.96338 (best_iter: 923)
  Fold 8: 0.96644 (best_iter: 922)
  Fold 9: 0.96451 (best_iter: 904)
  LightGBM OOF: 0.96572


## CatBoost

In [4]:
cat_features_idx = [feature_cols.index(c) for c in CAT_COLS]

oof_cat = np.zeros((len(X), 3))
test_cat = np.zeros((len(X_test), 3))

print("=== CatBoost ===")
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    model = CatBoostClassifier(
        iterations=2000,
        depth=8,
        learning_rate=0.04,
        l2_leaf_reg=4.0,
        loss_function='MultiClass',
        eval_metric='MultiClass',
        cat_features=cat_features_idx,
        auto_class_weights='Balanced',
        task_type='GPU',
        random_seed=SEED,
        verbose=0,
        early_stopping_rounds=150,
    )
    model.fit(X.iloc[tr_idx], y[tr_idx],
              eval_set=(X.iloc[va_idx], y[va_idx]))
    oof_cat[va_idx] = model.predict_proba(X.iloc[va_idx])
    test_cat += model.predict_proba(X_test) / N_FOLDS
    score = balanced_accuracy_score(y[va_idx], oof_cat[va_idx].argmax(axis=1))
    best_iter = model.get_best_iteration()
    print(f"  Fold {fold}: {score:.5f} (best_iter: {best_iter})")

cat_oof_score = balanced_accuracy_score(y, oof_cat.argmax(axis=1))
print(f"  CatBoost OOF: {cat_oof_score:.5f}")

=== CatBoost ===
  Fold 0: 0.96783 (best_iter: 866)
  Fold 1: 0.96609 (best_iter: 834)
  Fold 2: 0.96830 (best_iter: 728)
  Fold 3: 0.96974 (best_iter: 1053)
  Fold 4: 0.96966 (best_iter: 1004)
  Fold 5: 0.96822 (best_iter: 1021)
  Fold 6: 0.96788 (best_iter: 1007)
  Fold 7: 0.96854 (best_iter: 922)
  Fold 8: 0.96736 (best_iter: 898)
  Fold 9: 0.96833 (best_iter: 849)
  CatBoost OOF: 0.96820


## 개별 모델 비교 + 가중 블렌딩

In [5]:
print("=== 개별 모델 OOF ===")
print(f"  XGBoost:  {xgb_oof_score:.5f}")
print(f"  LightGBM: {lgb_oof_score:.5f}")
print(f"  CatBoost: {cat_oof_score:.5f}")

# 최적 가중치 탐색 (그리드 서치)
best_score = 0
best_weights = None

for w_xgb in np.arange(0.1, 0.8, 0.05):
    for w_lgb in np.arange(0.1, 0.8, 0.05):
        w_cat = 1.0 - w_xgb - w_lgb
        if w_cat < 0.05:
            continue
        blend = w_xgb * oof_xgb + w_lgb * oof_lgb + w_cat * oof_cat
        score = balanced_accuracy_score(y, blend.argmax(axis=1))
        if score > best_score:
            best_score = score
            best_weights = (w_xgb, w_lgb, w_cat)

w_xgb, w_lgb, w_cat = best_weights
print(f"\n=== 최적 블렌딩 ===")
print(f"  가중치: XGB={w_xgb:.2f}, LGB={w_lgb:.2f}, CAT={w_cat:.2f}")
print(f"  Blend OOF: {best_score:.5f}")

=== 개별 모델 OOF ===
  XGBoost:  0.96809
  LightGBM: 0.96572
  CatBoost: 0.96820

=== 최적 블렌딩 ===
  가중치: XGB=0.45, LGB=0.10, CAT=0.45
  Blend OOF: 0.96893


## 제출 파일 생성

In [6]:
# 개별 모델 + 블렌딩 제출
test_blend = w_xgb * test_xgb + w_lgb * test_lgb + w_cat * test_cat

submissions = {
    'exp004_xgb': test_xgb,
    'exp004_lgb': test_lgb,
    'exp004_cat': test_cat,
    'exp004_blend': test_blend,
}

for name, preds in submissions.items():
    labels = target_le.inverse_transform(preds.argmax(axis=1))
    sub = pd.DataFrame({'id': test['id'], 'Irrigation_Need': labels})
    path = f'../submissions/submission_{name}.csv'
    sub.to_csv(path, index=False)
    dist = sub['Irrigation_Need'].value_counts()
    print(f"{path}")
    print(f"  {dict(dist)}\n")

../submissions/submission_exp004_xgb.csv
  {'Low': np.int64(159866), 'Medium': np.int64(101187), 'High': np.int64(8947)}

../submissions/submission_exp004_lgb.csv
  {'Low': np.int64(159769), 'Medium': np.int64(101391), 'High': np.int64(8840)}

../submissions/submission_exp004_cat.csv
  {'Low': np.int64(159597), 'Medium': np.int64(100617), 'High': np.int64(9786)}

../submissions/submission_exp004_blend.csv
  {'Low': np.int64(159849), 'Medium': np.int64(101092), 'High': np.int64(9059)}

